In [ ]:
import os
import glob
import re
import xarray as xr
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
# consolidate runs
run_dir = "xbeach_cases"
output_filename = "consolidated_sediment_transport.nc"

nc_files = glob.glob(os.path.join(run_dir, "run_tp_*_ang_*", "xboutput.nc"))
print(f"Found {len(nc_files)} output files to process.")

datasets = []

for nc_path in nc_files:
    folder_name = os.path.basename(os.path.dirname(nc_path))
    match = re.search(r"run_tp_([\d\.]+)_ang_([\d\.-]+)", folder_name)
    
    if not match:
        print(f"Skipping folder (could not parse parameters): {folder_name}")
        continue
        
    tp_val = float(match.group(1))
    ang_val = float(match.group(2))
    
    with xr.open_dataset(nc_path, chunks={"globaltime": 100}) as ds:
        ds_filtered = ds[["Svsg", "Svbg"]]
        total_sed = ds_filtered["Svsg"] + ds_filtered["Svbg"]
        ds_case = total_sed.to_dataset(name="total_sediment_transport")
        ds_case = ds_case.assign_coords(tp=tp_val, mainang=ang_val)
        ds_case = ds_case.expand_dims(["tp", "mainang"])
        
        datasets.append(ds_case)

print("Merging...")
master_ds = xr.combine_by_coords(datasets)

print(f"Saving to {output_filename}...")
master_ds.to_netcdf(output_filename)
print('All done o.O')

In [ ]:
# averaging time and y axes, integrating over x
ds = xr.open_dataset("consolidated_sediment_transport.nc")

transport_time_avg = ds["total_sediment_transport"].mean(dim="globaltime")

y_mask = (ds["globaly"] >= 800) & (ds["globaly"] <= 1200)
transport_y_filtered = transport_time_avg.where(y_mask, drop=True)
profile_1d = transport_y_filtered.mean(dim="ny")

x_coord_1d = ds["globalx"].where(y_mask, drop=True).mean(dim="ny")
profile_1d = profile_1d.assign_coords(x_coord=("nx", x_coord_1d.data))
q_tot_da = profile_1d.integrate(coord="x_coord")

df_results = q_tot_da.to_dataframe(name="q_tot").reset_index()
df_results = df_results[['tp', 'mainang', 'q_tot']]

print("\n--- Final Integrated Transport (q_tot) per Case ---")
print(df_results)

#df_results.to_csv("q_tot_results.csv", index=False)

In [ ]:
# plotting
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available() else 'default')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6), sharex=True, sharey=True)

vmin = df_results['q_tot'].min()
vmax = df_results['q_tot'].max()

sc = ax1.scatter(
    df_results['Tp'], 
    df_results['mainang'], 
    c=df_results['q_tot'], 
    cmap='viridis', 
    s=180,               
    edgecolors='black',   
    linewidths=1.2,
    vmin=vmin,
    vmax=vmax
)
ax1.set_title("Discrete Wave Cases ($q_{tot}$)", fontsize=14, pad=12, weight='bold')
ax1.set_xlabel("Wave Period, $T_p$ (s)", fontsize=12, labelpad=8)
ax1.set_ylabel("Wave Angle, $mainang$ ($^\circ$)", fontsize=12, labelpad=8)
ax1.tick_params(labelsize=11)
ax1.grid(True, linestyle='--', alpha=0.6)

cf = ax2.tricontourf(
    df_results['Tp'], 
    df_results['mainang'], 
    df_results['q_tot'], 
    levels=200,           
    cmap='viridis',
    vmin=vmin,
    vmax=vmax
)

ax2.scatter(df_results['Tp'], df_results['mainang'], color='black', s=15, alpha=0.4, label='Run Locations')

ax2.set_title("Interpolated Metamodel Surface", fontsize=14, pad=12, weight='bold')
ax2.set_xlabel("Wave Period, $T_p$ (s)", fontsize=12, labelpad=8)
ax2.tick_params(labelsize=11)
ax2.grid(True, linestyle='--', alpha=0.6)

cbar = fig.colorbar(sc, ax=[ax1, ax2], orientation='vertical', fraction=0.046, pad=0.04)
cbar.set_label("Total Integrated Transport ($q_{tot}$)", fontsize=13, labelpad=12, weight='bold')
cbar.ax.tick_params(labelsize=11)

plt.tight_layout()
plt.show()